# 01 — Exploratory Data Analysis
**Brugada-HUCA ECG Dataset** | PhysioNet `brugada-huca/1.0.0`

Goals:
1. Inspect the metadata and class distribution
2. Load and sanity-check raw WFDB signals
3. Visualise ECG morphology differences between **Brugada** and **Normal** classes
4. Focus on leads **V1 / V2 / V3** — where the coved ST-elevation pattern appears
5. Identify signal-quality issues (noise, baseline drift, artefacts)

## 0 — Setup

In [ ]:
import sys
from pathlib import Path

# Locate project root regardless of where Jupyter launched from.
# Walks up from cwd until it finds the folder that contains src/config.py.
_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / 'src' / 'config.py').exists()),
    _cwd,
)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import wfdb

import config as cfg
from preprocessing import load_record, list_records, load_all_records

sns.set_theme(style='whitegrid', font_scale=1.1)
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

print('ROOT :', ROOT)
print('DATA :', cfg.DATA_RAW)
print('Data dir exists:', cfg.DATA_RAW.exists())

## 1 — Metadata Inspection

In [ ]:
meta = pd.read_csv(cfg.METADATA_FILE)
print(f'Shape: {meta.shape}')
meta.head(10)

In [ ]:
print('Column dtypes:')
print(meta.dtypes)
print('\nMissing values:')
print(meta.isnull().sum())

In [ ]:
print('Label distribution:')
print(meta[cfg.LABEL_COL].value_counts())
print(f"\nImbalance ratio: {meta[cfg.LABEL_COL].value_counts().iloc[0] / meta[cfg.LABEL_COL].value_counts().iloc[1]:.2f}:1")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Class distribution
counts = meta[cfg.LABEL_COL].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2196F3', '#F44336'], edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, (cls, cnt) in enumerate(counts.items()):
    axes[0].text(i, cnt + 2, str(cnt), ha='center', fontweight='bold')

# Basal pattern (if column exists)
if 'basal_pattern' in meta.columns:
    bp = meta.groupby([cfg.LABEL_COL, 'basal_pattern']).size().unstack(fill_value=0)
    bp.plot(kind='bar', ax=axes[1], colormap='Set2', edgecolor='black')
    axes[1].set_title('Basal Pattern by Class')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=0)
else:
    axes[1].text(0.5, 0.5, 'basal_pattern\nnot available', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Basal Pattern')

# Sudden death flag (if column exists)
if 'sudden_death' in meta.columns:
    sd = meta.groupby([cfg.LABEL_COL, 'sudden_death']).size().unstack(fill_value=0)
    sd.plot(kind='bar', ax=axes[2], colormap='Set1', edgecolor='black')
    axes[2].set_title('Sudden Death Flag by Class')
    axes[2].set_xlabel('')
    axes[2].tick_params(axis='x', rotation=0)
else:
    axes[2].text(0.5, 0.5, 'sudden_death\nnot available', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('Sudden Death')

plt.tight_layout()
plt.savefig(REPORTS / 'eda_class_distribution.png', dpi=150)
plt.show()

## 2 — Record Inventory

In [ ]:
records = list_records(cfg.DATA_RAW)
print(f'Records in RECORDS file: {len(records)}')
print('First 10:', records[:10])

In [ ]:
# Cross-check: all patient IDs in metadata have a corresponding WFDB record
meta_ids = set(meta[cfg.PATIENT_ID_COL].astype(str))
record_ids = set(records)
print(f'In metadata, missing from RECORDS: {meta_ids - record_ids}')
print(f'In RECORDS, missing from metadata: {record_ids - meta_ids}')

## 3 — Load a Sample Record

In [ ]:
# Pick one Brugada and one Normal patient
# brugada column: 0=Normal, 1=Type-1 Brugada, 2=Type-2 Brugada — both types are positive
brugada_ids = meta.loc[meta[cfg.LABEL_COL] >= 1, cfg.PATIENT_ID_COL].astype(str).tolist()
normal_ids  = meta.loc[meta[cfg.LABEL_COL] == 0, cfg.PATIENT_ID_COL].astype(str).tolist()

sample_b = brugada_ids[0]
sample_n = normal_ids[0]

sig_b, _ = load_record(sample_b, cfg.DATA_RAW)
sig_n, _ = load_record(sample_n, cfg.DATA_RAW)

print(f'Brugada  ({sample_b}): shape={sig_b.shape}, min={sig_b.min():.3f}, max={sig_b.max():.3f} mV')
print(f'Normal   ({sample_n}): shape={sig_n.shape}, min={sig_n.min():.3f}, max={sig_n.max():.3f} mV')

## 4 — Full 12-Lead ECG Visualisation

In [ ]:
def plot_12lead(signal, title, fs=cfg.FS, color='steelblue', save_path=None):
    """Plot all 12 leads in a standard clinical layout."""
    time = np.arange(signal.shape[1]) / fs
    fig, axes = plt.subplots(6, 2, figsize=(16, 14), sharex=True)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)

    for i, (ax, lead) in enumerate(zip(axes.ravel(), cfg.LEAD_NAMES)):
        ax.plot(time, signal[i], color=color, linewidth=0.8)
        ax.set_ylabel(lead, fontsize=10, rotation=0, labelpad=24)
        ax.yaxis.set_label_position('left')
        ax.set_ylim(signal[i].min() - 0.1, signal[i].max() + 0.1)

        # Highlight if this is a Brugada-specific lead
        if lead in cfg.BRUGADA_LEADS:
            ax.set_facecolor('#FFF3E0')
            ax.set_ylabel(f'{lead} *', fontsize=10, rotation=0, labelpad=24, color='darkorange')

    axes[-1, 0].set_xlabel('Time (s)')
    axes[-1, 1].set_xlabel('Time (s)')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_12lead(sig_b, f'Brugada Patient — ID: {sample_b}', color='#C62828',
            save_path=REPORTS / 'eda_ecg_brugada_sample.png')
plot_12lead(sig_n, f'Normal Patient — ID: {sample_n}', color='#1565C0',
            save_path=REPORTS / 'eda_ecg_normal_sample.png')

## 5 — V1 / V2 / V3 Side-by-Side Comparison
These leads show the pathognomonic **coved ST-elevation** in Brugada syndrome.

In [ ]:
def plot_brugada_leads(sig_brugada, sig_normal, id_b, id_n, n_examples=3, save_path=None):
    """Overlay V1/V2/V3 for Brugada vs Normal, showing multiple examples."""
    time = np.arange(cfg.N_SAMPLES) / cfg.FS
    brugada_lead_idx = [cfg.LEAD_NAMES.index(l) for l in cfg.BRUGADA_LEADS]

    fig, axes = plt.subplots(3, 2, figsize=(16, 9), sharex=True)
    fig.suptitle('Brugada vs Normal — Leads V1, V2, V3 (orange background)', fontsize=13, fontweight='bold')

    for row, (li, lead_name) in enumerate(zip(brugada_lead_idx, cfg.BRUGADA_LEADS)):
        ax_b = axes[row, 0]
        ax_n = axes[row, 1]

        ax_b.plot(time, sig_brugada[li], color='#C62828', linewidth=1.2)
        ax_b.set_ylabel(lead_name, fontweight='bold')
        ax_b.set_facecolor('#FFF3E0')
        if row == 0:
            ax_b.set_title(f'BRUGADA (ID: {id_b})', color='#C62828', fontweight='bold')

        ax_n.plot(time, sig_normal[li], color='#1565C0', linewidth=1.2)
        ax_n.set_facecolor('#E3F2FD')
        if row == 0:
            ax_n.set_title(f'NORMAL (ID: {id_n})', color='#1565C0', fontweight='bold')

    axes[-1, 0].set_xlabel('Time (s)')
    axes[-1, 1].set_xlabel('Time (s)')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_brugada_leads(sig_b, sig_n, sample_b, sample_n,
                   save_path=REPORTS / 'eda_v1v2v3_comparison.png')

## 6 — Multi-Sample Comparison (5 Brugada vs 5 Normal) in V1

In [ ]:
N_SHOW = 5
time = np.arange(cfg.N_SAMPLES) / cfg.FS
v1_idx = cfg.LEAD_NAMES.index('V1')

fig, axes = plt.subplots(N_SHOW, 2, figsize=(16, 12), sharex=True, sharey=False)
fig.suptitle('Lead V1 — Brugada (left) vs Normal (right)', fontsize=13, fontweight='bold')

for i in range(N_SHOW):
    sig_b_i, _ = load_record(brugada_ids[i], cfg.DATA_RAW)
    sig_n_i, _ = load_record(normal_ids[i], cfg.DATA_RAW)

    axes[i, 0].plot(time, sig_b_i[v1_idx], color='#C62828', linewidth=0.9)
    axes[i, 0].set_ylabel(f'B-{brugada_ids[i]}', fontsize=8, rotation=0, labelpad=36)
    axes[i, 0].set_facecolor('#FFF3E0')

    axes[i, 1].plot(time, sig_n_i[v1_idx], color='#1565C0', linewidth=0.9)
    axes[i, 1].set_ylabel(f'N-{normal_ids[i]}', fontsize=8, rotation=0, labelpad=36)
    axes[i, 1].set_facecolor('#E3F2FD')

    if i == 0:
        axes[i, 0].set_title('BRUGADA', color='#C62828', fontweight='bold')
        axes[i, 1].set_title('NORMAL', color='#1565C0', fontweight='bold')

axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(REPORTS / 'eda_v1_multisample.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 — Signal Statistics Per Lead

In [ ]:
print('Loading all records for statistical analysis...')
X_all, y_all, ids_all = load_all_records(meta, cfg.DATA_RAW, verbose=True)
print(f'Loaded: X={X_all.shape}, y={y_all.shape}')
print(f'Brugada: {(y_all==1).sum()} | Normal: {(y_all==0).sum()}')

In [ ]:
# Mean amplitude per lead, split by class
X_b = X_all[y_all == 1]  # Brugada signals
X_n = X_all[y_all == 0]  # Normal signals

# Mean absolute amplitude per lead (mV)
amp_b = np.abs(X_b).mean(axis=(0, 2))  # (12,)
amp_n = np.abs(X_n).mean(axis=(0, 2))  # (12,)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(cfg.N_LEADS)
width = 0.35
bars_b = ax.bar(x - width/2, amp_b, width, label='Brugada', color='#C62828', alpha=0.85, edgecolor='black')
bars_n = ax.bar(x + width/2, amp_n, width, label='Normal',  color='#1565C0', alpha=0.85, edgecolor='black')

# Highlight V1/V2/V3
for li, ln in enumerate(cfg.LEAD_NAMES):
    if ln in cfg.BRUGADA_LEADS:
        ax.axvspan(li - 0.6, li + 0.6, alpha=0.08, color='orange', zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(cfg.LEAD_NAMES)
ax.set_ylabel('Mean |Amplitude| (mV)')
ax.set_title('Mean Absolute Amplitude per Lead — Brugada vs Normal')
ax.legend()
ax.text(0.98, 0.95, '* Orange shading = V1/V2/V3', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color='darkorange')
plt.tight_layout()
plt.savefig(REPORTS / 'eda_amplitude_per_lead.png', dpi=150)
plt.show()

In [ ]:
# Mean ± std waveform envelope for V1 — Brugada vs Normal
v1_idx = cfg.LEAD_NAMES.index('V1')
time = np.arange(cfg.N_SAMPLES) / cfg.FS

mean_b = X_b[:, v1_idx, :].mean(axis=0)
std_b  = X_b[:, v1_idx, :].std(axis=0)
mean_n = X_n[:, v1_idx, :].mean(axis=0)
std_n  = X_n[:, v1_idx, :].std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=True)

for ax, mean, std, label, color in [
    (axes[0], mean_b, std_b, 'Brugada', '#C62828'),
    (axes[1], mean_n, std_n, 'Normal',  '#1565C0'),
]:
    ax.fill_between(time, mean - std, mean + std, alpha=0.25, color=color)
    ax.plot(time, mean, color=color, linewidth=1.5, label=label)
    ax.set_title(f'Lead V1 — {label} (mean ± 1 SD, n={len(X_b if label=="Brugada" else X_n)})')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (mV)')
    ax.set_facecolor('#FFF3E0' if label == 'Brugada' else '#E3F2FD')

plt.suptitle('Mean Waveform Envelope — Lead V1', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'eda_v1_mean_envelope.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 — Signal Quality Check

In [ ]:
# Check for flat signals (zero variance) and extreme amplitudes
issues = []
for i, (sig, pid) in enumerate(zip(X_all, ids_all)):
    for li, lead in enumerate(cfg.LEAD_NAMES):
        std = sig[li].std()
        amax = np.abs(sig[li]).max()
        if std < 0.005:
            issues.append({'patient': pid, 'lead': lead, 'issue': 'flat signal', 'value': round(float(std), 6)})
        elif amax > 5.0:
            issues.append({'patient': pid, 'lead': lead, 'issue': 'extreme amplitude', 'value': round(float(amax), 3)})

if issues:
    df_issues = pd.DataFrame(issues)
    print(f'Found {len(df_issues)} signal quality issues:')
    display(df_issues)
else:
    print('No signal quality issues detected.')

In [ ]:
# Amplitude distribution per class (all leads combined)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(X_b.ravel(), bins=200, density=True, alpha=0.6, color='#C62828', label='Brugada')
ax.hist(X_n.ravel(), bins=200, density=True, alpha=0.6, color='#1565C0', label='Normal')
ax.set_xlabel('Amplitude (mV)')
ax.set_ylabel('Density')
ax.set_title('Amplitude Distribution — All Leads, All Patients')
ax.set_xlim(-3, 3)
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS / 'eda_amplitude_distribution.png', dpi=150)
plt.show()

## 9 — EDA Summary

| Observation | Finding |
|---|---|
| Total records | 363 (76 Brugada, 287 Normal) |
| Class imbalance | ~1:4 → requires SMOTE or `pos_weight` in loss |
| Signal format | 12 leads × 1200 samples @ 100 Hz = 12 sec |
| Key diagnostic leads | V1, V2, V3 (coved ST elevation visible) |
| Signal quality | Check output of cell 8 above |
| Preprocessing needs | Bandpass (0.5–40 Hz) + Notch (50 Hz) + Z-score normalization |

**Next step:** `02_preprocessing.ipynb` — apply the full preprocessing pipeline and create train/val/test splits.